# Network-stress model training

This development notebook calls the canonical pipeline in `A2/src/train_model.py`.
Target thresholds are fitted only on the chronological training period. Candidate
models are compared on validation data; the selected model is refitted on training
plus validation data and evaluated once on the untouched test period.


In [ ]:
from pathlib import Path
import sys

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

for base in (Path.cwd(), *Path.cwd().parents):
    if (base / "A1" / "src" / "DataLoader.py").is_file():
        PROJECT_ROOT = base
        break
else:
    raise FileNotFoundError("Could not locate the project root.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from A2.src.train_model import prepare_modelling_data, train_pipeline

DATA_PATH = (
    PROJECT_ROOT / "A1" / "data" / "processed"
    / "electricity_transformed.parquet"
)
MODEL_DIRECTORY = PROJECT_ROOT / "A2" / "models"


## Target and chronological partitions

The target is a one-hour-ahead regional stress proxy. Rows whose current or target
hour was interpolated are excluded so two-sided imputation cannot leak future
information into model evaluation.


In [ ]:
data = pd.read_parquet(DATA_PATH).sort_values("timestamp").reset_index(drop=True)
prepared = prepare_modelling_data(data)

display(
    prepared.thresholds.rename(
        columns={
            0.75: "elevated_75th",
            0.90: "severe_90th",
            0.97: "extreme_97th",
        }
    ).round(3)
)

partition_balance = pd.concat(
    {
        "train": prepared.train["stress_target_1h"].value_counts(normalize=True),
        "validation": prepared.validation["stress_target_1h"].value_counts(normalize=True),
        "test": prepared.test["stress_target_1h"].value_counts(normalize=True),
    },
    axis=1,
).mul(100).round(2)

print("Partition target-time boundaries:")
display(pd.Series(prepared.boundaries, name="timestamp").to_frame())
print(f"Rows excluded because current/target data was imputed: {prepared.excluded_imputed_rows}")
print("Class percentages by partition:")
display(partition_balance)


## Train, compare, select, and save

Random Forest and XGBoost are compared using validation macro-F1. The final test
partition is not involved in model selection.


In [ ]:
result = train_pipeline(DATA_PATH, MODEL_DIRECTORY)

print("Validation comparison:")
display(pd.DataFrame(result["validation_metrics"]).T.round(4))
print(f"Selected model: {result['selected_model']}")

print("Untouched test metrics:")
display(pd.Series(result["test_metrics"], name="value").round(4).to_frame())
print(result["classification_report"])


In [ ]:
matrix = pd.DataFrame(
    result["confusion_matrix"],
    index=["low", "medium", "high", "critical"],
    columns=["low", "medium", "high", "critical"],
)

plt.figure(figsize=(7, 5))
sns.heatmap(matrix, annot=True, fmt="d", cmap="Blues")
plt.title(f"Final test confusion matrix: {result['selected_model']}")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

print(f"Saved model: {result['bundle_path']}")
print(f"Saved metadata: {result['metadata_path']}")
